# Curvas ROC y figuras comparativas del artículo

**Objetivo:** ROC, SHAP comparativo y cuatro indicadores; reentrena modelos.

Este cuaderno contiene el código completo del experimento `generate_ieee_figures.py`,
dividido en etapas y acompañado de explicaciones. No necesita ZIP, clonar un
repositorio ni ejecutar un script oculto. Ejecuta las celdas en orden.

**Recorrido:** entorno → carga e inspección → preparación → modelo o análisis →
evaluación → gráficos → descarga y colaboración.

**Entradas:** `dataset_entrenamiento_sierra_realista.csv`.

Los datos no se incluyen. Se conservan los valores predeterminados del protocolo histórico.
Las salidas se generarán al ejecutar; las figuras históricas no están incrustadas.


**Interpretación:** los pesos de atención, la importancia Gini y SHAP son indicadores distintos. No equivalen a causas físicas. El módulo de atención se conserva para documentar el experimento; su revisión debe evaluarse como una variante nueva.

## 1. Entorno y parámetros
Modifica los parámetros aquí para que una colaboración sea fácil de revisar. Los valores originales siguen seleccionados. Cada ejecución crea una carpeta nueva.

In [ ]:
# Instala únicamente las bibliotecas que falten en el entorno.
import importlib.util
import importlib.metadata
import subprocess
import sys
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image

DEPENDENCIAS = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn', 'matplotlib': 'matplotlib', 'torch': 'torch', 'shap': 'shap'}
faltantes = [paquete for modulo, paquete in DEPENDENCIAS.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *faltantes], check=True)

CONFIG = dict(seed=42, rf_trees=100, rf_depth=15, epochs=50,
              learning_rate=0.002, batch_size=256)
EXPERIMENTO = 'figuras_articulo'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DATA_DIR = Path(globals().get('DATA_DIR_OVERRIDE', 'data')).resolve()
OUTPUT_DIR = Path(globals().get('OUTPUT_DIR_OVERRIDE', Path('results') / EXPERIMENTO / RUN_ID)).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLBACKEND', 'Agg')
print('Parámetros:', CONFIG)
print('Salidas:', OUTPUT_DIR)

def mostrar_figura(ruta):
    # Conserva la figura en disco y también la muestra en el cuaderno.
    if str(ruta).lower().endswith('.png'):
        display(Image(filename=str(ruta)))


## 2. Carga e inspección de los datos
En Colab, la siguiente celda ofrece el selector si faltan archivos. Carga solo los CSV indicados, conservando sus nombres. En un entorno local, colócalos en `data/`. Para el dataset realista usa la copia de la subcarpeta `resultados` original.

In [ ]:
import pandas as pd

ARCHIVOS_REQUERIDOS = ['dataset_entrenamiento_sierra_realista.csv']
def localizar(nombre):
    candidatos = [DATA_DIR / nombre, DATA_DIR / 'resultados' / nombre]
    return next((p for p in candidatos if p.is_file()), candidatos[0])

pendientes = [n for n in ARCHIVOS_REQUERIDOS if not localizar(n).is_file()]
if pendientes:
    print('Archivos requeridos:', pendientes)
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Coloca los CSV en ' + str(DATA_DIR))
    cargados = files.upload()
    for nombre, contenido in cargados.items():
        if Path(nombre).name != nombre or nombre not in ARCHIVOS_REQUERIDOS:
            raise ValueError('Archivo inesperado: ' + nombre)
        (DATA_DIR / nombre).write_bytes(contenido)
    del cargados

DATA_FILES = {n: localizar(n) for n in ARCHIVOS_REQUERIDOS}
if any(not p.is_file() for p in DATA_FILES.values()):
    raise FileNotFoundError('Faltan CSV; repite esta celda y carga los indicados.')

# Inspección visible: dimensiones, tipos, valores faltantes y balance de clases.
for nombre, ruta in DATA_FILES.items():
    tabla = pd.read_csv(ruta)
    print('\n', nombre, '| filas y columnas:', tabla.shape)
    display(tabla.head())
    display(pd.DataFrame({'tipo': tabla.dtypes.astype(str), 'nulos': tabla.isna().sum()}))
    if 'clase' in tabla.columns:
        display(tabla['clase'].value_counts(dropna=False).rename('filas').to_frame())

REGISTRO = dict(experimento=EXPERIMENTO, parametros=CONFIG, run_id=RUN_ID,
    input_sha256={n: hashlib.sha256(p.read_bytes()).hexdigest() for n, p in DATA_FILES.items()},
    versiones={p: importlib.metadata.version(p) for p in DEPENDENCIAS.values()},
    estado='iniciado')
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 9.5
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8.5
plt.rcParams['ytick.labelsize'] = 8.5
plt.rcParams['legend.fontsize'] = 8.5
plt.rcParams['figure.titlesize'] = 10.5
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.05
BASE_DIR = str(OUTPUT_DIR)
OUT_DIR = str(OUTPUT_DIR)
FILE_DATASET = str(DATA_FILES['dataset_entrenamiento_sierra_realista.csv'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])


### Definición: `FeatureSelfAttention`
Esta definición queda visible para revisar su arquitectura y proponer variantes. Sus valores originales se conservan.

In [ ]:
class FeatureSelfAttention(nn.Module):
    def __init__(self, input_dim):
        super(FeatureSelfAttention, self).__init__()
        self.query = nn.Linear(input_dim, input_dim, bias=False)
        self.key = nn.Linear(input_dim, input_dim, bias=False)
        self.value = nn.Linear(input_dim, input_dim, bias=False)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        attention_scores = torch.matmul(q.unsqueeze(2), k.unsqueeze(1)) / np.sqrt(x.shape[1])
        attention_weights = self.softmax(attention_scores.mean(dim=2))
        attended_x = v * attention_weights
        return attended_x, attention_weights


### Definición: `FireAttentionMLP`
Esta definición queda visible para revisar su arquitectura y proponer variantes. Sus valores originales se conservan.

In [ ]:
class FireAttentionMLP(nn.Module):
    def __init__(self, input_dim):
        super(FireAttentionMLP, self).__init__()
        self.attention = FeatureSelfAttention(input_dim)
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        attended_x, weights = self.attention(x)
        output = self.classifier(attended_x)
        return output, weights


## 4. Desarrollo del experimento
Las siguientes celdas contienen el cuerpo del análisis original, con comentarios y código editable. Puedes inspeccionar las variables entre pasos. Ejecuta nuevamente desde la carga de datos si cambias una transformación previa.

### Etapa 1: Partición y preparación de muestras
La división aleatoria usa una semilla fija. El escalador se ajusta solo sobre entrenamiento. Una partición aleatoria no demuestra transferencia geográfica.

In [ ]:
print("="*75)
print("  GENERANDO FIGURAS FALTANTES PARA EL ARTÍCULO IEEE (600 DPI PNG + PDF)")
print("="*75)

if not os.path.exists(FILE_DATASET):
    print(f"[ERROR] No se encuentra el dataset: {FILE_DATASET}")
    raise ValueError("Revisa el archivo y las columnas indicadas en la salida anterior.")
df = pd.read_csv(FILE_DATASET).dropna()
features = ['elevacion', 'pendiente', 'orientacion', 'ndvi', 'evi', 'nbr', 'temp_c', 'precip_mm']

nombres_es = {
    'elevacion': 'Elevación',
    'pendiente': 'Pendiente',
    'orientacion': 'Orientación',
    'temp_c': 'Temperatura',
    'precip_mm': 'Precipitación',
    'ndvi': 'NDVI',
    'evi': 'EVI',
    'nbr': 'NBR'
}

X = df[features].values
y = df['clase'].values

# Misma división exacta 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=CONFIG['seed'], stratify=y)


### Etapa 2: Entrenamiento de la red
Se optimizan los pesos con los datos de entrenamiento. El registro muestra la pérdida de la última tanda; no es una métrica de validación. Mantén fijo el conjunto de prueba al comparar configuraciones.

In [ ]:

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Entrenar Random Forest
rf = RandomForestClassifier(n_estimators=CONFIG['rf_trees'], max_depth=CONFIG['rf_depth'], random_state=CONFIG['seed'], n_jobs=-1)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_proba)
rf_gini = rf.feature_importances_

# Entrenar Feature-Attention MLP
model = FireAttentionMLP(input_dim=len(features))
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-5)

ds = TensorDataset(torch.FloatTensor(X_train_s), torch.FloatTensor(y_train).unsqueeze(1))
ld = DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=True)

model.train()
for epoca in range(CONFIG['epochs']):
    for bx, by in ld:
        optimizer.zero_grad()
        out, _ = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
    if epoca == 0 or (epoca + 1) % 10 == 0 or epoca + 1 == CONFIG['epochs']:
        print(f"Época {epoca + 1}/{CONFIG['epochs']} | pérdida última tanda: {float(loss.detach()):.5f}")


### Etapa 3: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:

model.eval()
X_test_t = torch.FloatTensor(X_test_s)
with torch.no_grad():
    nn_proba_t, nn_weights_t = model(X_test_t)
    nn_proba = nn_proba_t.numpy().flatten()
    nn_auc = roc_auc_score(y_test, nn_proba)
    attn_weights = nn_weights_t.mean(dim=0).numpy()
    attn_weights_norm = attn_weights / np.sum(attn_weights)

# =========================================================================
# FIGURA 1: CURVA ROC COMPARATIVA (IEEE STYLE)
# =========================================================================
print(f"\n[1/3] Generando Figura 1: Curva ROC comparativa (RF AUC={rf_auc:.4f}, Attention MLP AUC={nn_auc:.4f})...")

fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
fpr_nn, tpr_nn, _ = roc_curve(y_test, nn_proba)

fig, ax = plt.subplots(figsize=(4.5, 3.8), dpi=600)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

ax.plot(fpr_rf, tpr_rf, color='#d95f0e', lw=1.8, label=f'Random Forest (AUC = {rf_auc:.4f})')
ax.plot(fpr_nn, tpr_nn, color='#2b8cbe', lw=1.8, label=f'Feature-Attention MLP (AUC = {nn_auc:.4f})')
ax.plot([0, 1], [0, 1], color='#777777', lw=1.0, linestyle='--', label='Random Chance (AUC = 0.5000)')

ax.set_xlim([-0.01, 1.01])


### Etapa 4: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:
ax.set_ylim([-0.01, 1.01])
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.set_title('ROC Curve Comparison', fontweight='bold', pad=8)
ax.grid(True, linestyle=':', alpha=0.5, color='#cccccc')
ax.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='#dddddd')

# Guardar PNG 600 DPI y PDF
fn_roc_png = os.path.join(OUT_DIR, "roc_rf_vs_attention_mlp.png")
fn_roc_pdf = os.path.join(OUT_DIR, "roc_rf_vs_attention_mlp.pdf")
plt.savefig(fn_roc_png, dpi=600, facecolor='white')
mostrar_figura(fn_roc_png)
plt.savefig(fn_roc_pdf, facecolor='white')
mostrar_figura(fn_roc_pdf)
plt.close()
print(f"  -> Guardado: {fn_roc_png}")
print(f"  -> Guardado: {fn_roc_pdf}")

# =========================================================================
# FIGURA 2: SHAP COMPARATIVO DOS PANELES (a) RF (b) Attention MLP
# =========================================================================
print("\n[2/3] Generando Figura 2: SHAP comparativo de 2 paneles (a) RF (b) Feature-Attention MLP...")

# Submuestra estables de test
test_idx = np.random.choice(len(X_test), 1000, replace=False)
X_test_sample = X_test[test_idx]


### Etapa 5: Atribuciones SHAP
Inspecciona la muestra explicada, el conjunto de referencia y la clase de salida. SHAP describe el comportamiento del modelo; no demuestra causalidad. KernelExplainer puede tardar varios minutos.

In [ ]:
X_test_sample_s = X_test_s[test_idx]

df_test_sample = pd.DataFrame(X_test_sample, columns=[nombres_es[f] for f in features])

# SHAP RF
explainer_rf = shap.TreeExplainer(rf)
shap_vals_rf = explainer_rf.shap_values(X_test_sample)
# Para clasificación binaria, tomamos la clase 1 (Incendio)
if isinstance(shap_vals_rf, list):
    shap_vals_rf = shap_vals_rf[1]
elif len(np.shape(shap_vals_rf)) == 3:
    shap_vals_rf = shap_vals_rf[:, :, 1]

# SHAP Attention MLP
def predict_nn(x_numpy):
    x_tensor = torch.FloatTensor(x_numpy)
    with torch.no_grad():
        out, _ = model(x_tensor)
        return out.numpy().flatten()

bg_idx = np.random.choice(len(X_train_s), 100, replace=False)
background = X_train_s[bg_idx]
explainer_nn = shap.KernelExplainer(predict_nn, background)

shap_idx = np.random.choice(len(X_test_sample_s), 300, replace=False)
shap_vals_nn = explainer_nn.shap_values(X_test_sample_s[shap_idx])


### Etapa 6: Atribuciones SHAP
Inspecciona la muestra explicada, el conjunto de referencia y la clase de salida. SHAP describe el comportamiento del modelo; no demuestra causalidad. KernelExplainer puede tardar varios minutos.

In [ ]:
if isinstance(shap_vals_nn, list):
    shap_vals_nn = shap_vals_nn[0]
if len(shap_vals_nn.shape) == 3:
    shap_vals_nn = shap_vals_nn.squeeze()

df_test_nn_sample = df_test_sample.iloc[shap_idx]

# Calcular orden único uniforme basado en magnitud media de SHAP en Attention Net
mean_shap_nn = np.abs(shap_vals_nn).mean(axis=0).ravel()
mean_shap_rf = np.abs(shap_vals_rf).mean(axis=0).ravel()

# Orden exacto para ambas figuras (de mayor a menor importancia)
order_idx = np.argsort(mean_shap_nn)[::-1]
feature_order = [features[i] for i in order_idx]
feature_names_order = [nombres_es[f] for f in feature_order]

# Crear figura de 2 paneles horizontales (a) y (b)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5), dpi=600)
fig.patch.set_facecolor('white')

# Subplot (a) RF
plt.sca(ax1)
shap.summary_plot(shap_vals_rf[:, order_idx], df_test_sample.iloc[:, order_idx], show=False, plot_size=None, sort=False)
ax1.set_title('(a) Random Forest SHAP Summary', fontweight='bold', fontsize=9.5, pad=6)
ax1.set_xlabel('SHAP value (impact on fire probability)', fontsize=8.5)

# Subplot (b) Attention MLP
plt.sca(ax2)


### Etapa 7: Atribuciones SHAP
Inspecciona la muestra explicada, el conjunto de referencia y la clase de salida. SHAP describe el comportamiento del modelo; no demuestra causalidad. KernelExplainer puede tardar varios minutos.

In [ ]:
shap.summary_plot(shap_vals_nn[:, order_idx], df_test_nn_sample.iloc[:, order_idx], show=False, plot_size=None, sort=False)
ax2.set_title('(b) Feature-Attention MLP SHAP Summary', fontweight='bold', fontsize=9.5, pad=6)
ax2.set_xlabel('SHAP value (impact on fire probability)', fontsize=8.5)

plt.tight_layout()

fn_shap_png = os.path.join(OUT_DIR, "shap_rf_vs_attention_mlp.png")
fn_shap_pdf = os.path.join(OUT_DIR, "shap_rf_vs_attention_mlp.pdf")
plt.savefig(fn_shap_png, dpi=600, facecolor='white')
mostrar_figura(fn_shap_png)
plt.savefig(fn_shap_pdf, facecolor='white')
mostrar_figura(fn_shap_pdf)
plt.close()
print(f"  -> Guardado: {fn_shap_png}")
print(f"  -> Guardado: {fn_shap_pdf}")

# =========================================================================
# FIGURA 3: COMPARACIÓN GLOBAL DE EXPLICABILIDAD (BARRAS AGRUPADAS 0-100%)
# =========================================================================
print("\n[3/3] Generando Figura 3: Comparación global de explicabilidad (Barras agrupadas 0-100%)...")

# 4 Métricas: RF-Gini, Attention Weights, SHAP RF, SHAP Attention MLP (Asegurar 1D de longitud 8)
raw_rf_gini = np.array(rf_gini).ravel()
raw_attn_weights = np.array(attn_weights_norm).ravel()
raw_shap_rf = np.array(mean_shap_rf).ravel()
raw_shap_nn = np.array(mean_shap_nn).ravel()


### Etapa 8: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:

print(f"DEBUG: raw_rf_gini shape={raw_rf_gini.shape}")
print(f"DEBUG: raw_attn_weights shape={raw_attn_weights.shape}")
print(f"DEBUG: raw_shap_rf shape={raw_shap_rf.shape}")
print(f"DEBUG: raw_shap_nn shape={raw_shap_nn.shape}")

# Normalizar individualmente a porcentaje 0 - 100%
norm_rf_gini = (raw_rf_gini / np.sum(raw_rf_gini)) * 100
norm_attn_weights = (raw_attn_weights / np.sum(raw_attn_weights)) * 100
norm_shap_rf = (raw_shap_rf / np.sum(raw_shap_rf)) * 100
norm_shap_nn = (raw_shap_nn / np.sum(raw_shap_nn)) * 100

# DataFrame ordenado por SHAP Attention Net
df_exp = pd.DataFrame({
    'feature': features,
    'nombre': [nombres_es[f] for f in features],
    'RF-Gini': norm_rf_gini,
    'Attention Weights': norm_attn_weights,
    'SHAP (RF)': norm_shap_rf,
    'SHAP (Attention MLP)': norm_shap_nn
}).sort_values(by='SHAP (Attention MLP)', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7.2, 4.8), dpi=600)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

y_indices = np.arange(len(df_exp))


### Etapa 9: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:
bar_height = 0.18

c1 = '#d95f0e' # RF Gini (Naranja oscuro)
c2 = '#fec44f' # SHAP RF (Naranja claro)
c3 = '#2b8cbe' # Attention Weights (Azul oscuro)
c4 = '#a6bddb' # SHAP Attention MLP (Azul claro)

rects1 = ax.barh(y_indices + 1.5*bar_height, df_exp['RF-Gini'], bar_height, label='RF-Gini', color=c1, edgecolor='none')
rects2 = ax.barh(y_indices + 0.5*bar_height, df_exp['SHAP (RF)'], bar_height, label='SHAP (Random Forest)', color=c2, edgecolor='none')
rects3 = ax.barh(y_indices - 0.5*bar_height, df_exp['Attention Weights'], bar_height, label='Attention Weights', color=c3, edgecolor='none')
rects4 = ax.barh(y_indices - 1.5*bar_height, df_exp['SHAP (Attention MLP)'], bar_height, label='SHAP (Feature-Attention MLP)', color=c4, edgecolor='none')

ax.set_yticks(y_indices)
ax.set_yticklabels(df_exp['nombre'], fontweight='semibold')
ax.set_xlabel('Normalized Importance / Attribution (%)', fontweight='bold')
ax.set_title('Comparison of feature importance and explainability methods', fontweight='bold', pad=10)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(True, linestyle=':', alpha=0.5, color='#cccccc', axis='x')

# Quitar bordes innecesarios para estética IEEE
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')

ax.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='#dddddd', fontsize=8)


### Etapa 10: Representación y exportación
Lee los ejes y el conjunto de datos utilizado. Las figuras PNG se muestran al guardarse; al final también puedes inspeccionar todas las salidas y descargarlas.

In [ ]:

plt.tight_layout()

fn_exp_png = os.path.join(OUT_DIR, "feature_importance_explainability_comparison.png")
fn_exp_pdf = os.path.join(OUT_DIR, "feature_importance_explainability_comparison.pdf")
plt.savefig(fn_exp_png, dpi=600, facecolor='white')
mostrar_figura(fn_exp_png)
plt.savefig(fn_exp_pdf, facecolor='white')
mostrar_figura(fn_exp_pdf)
plt.close()
print(f"  -> Guardado: {fn_exp_png}")
print(f"  -> Guardado: {fn_exp_pdf}")

print("\n" + "="*75)
print("  [ÉXITO] TODAS LAS FIGURAS FALTANTES FUERON GENERADAS CON ÉXITO")
print("="*75)


## 5. Inspeccionar y guardar resultados
Las tablas se muestran debajo. El registro identifica parámetros, versiones y hashes. Esta celda marca el registro como completado: ejecútala solo cuando hayan terminado todas las etapas anteriores.

In [ ]:
for ruta in sorted(OUTPUT_DIR.glob('*.csv')):
    print(ruta.name)
    display(pd.read_csv(ruta).head(20))
REGISTRO['estado'] = 'completado'
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')
print('Salidas:', [p.name for p in sorted(OUTPUT_DIR.iterdir())])


In [ ]:
# Descarga las salidas antes de cerrar una sesión de Colab.
import shutil
paquete = shutil.make_archive(str(OUTPUT_DIR.parent / (EXPERIMENTO + '-' + RUN_ID)), 'zip', OUTPUT_DIR)
try:
    from google.colab import files
except ImportError:
    print('ZIP guardado en:', paquete)
else:
    files.download(paquete)


## 6. Proponer una colaboración

1. Plantea una pregunta concreta: por ejemplo, el efecto de un hiperparámetro o de una variable.
2. Conserva una ejecución con la configuración original y luego cambia una sola decisión.
3. Registra el hash del dataset, la partición, la semilla y todas las métricas relevantes.
4. En GitHub, abre un Issue para discutir la propuesta. Trabaja en un fork o en una rama
   con permisos y envía un pull request que explique el cambio y su evidencia.
5. Antes del commit, borra las salidas del cuaderno y revisa que no incluyan datos privados.
   Adjunta resultados únicamente cuando tengas permiso para compartirlos.

Edita la siguiente celda Markdown para documentar tu propuesta. No se envía nada automáticamente.

### Registro de la propuesta

- **Autor y objetivo:**
- **Hipótesis:**
- **Parámetro o código modificado:**
- **Dataset y hash:**
- **Semilla y partición:**
- **Resultado original y resultado de la variante:**
- **Limitaciones y conclusión:**
